### --creting the extrnal table to  acces the data in powre Bi use default schema, and creted the external table

In [0]:
%sql
create external table if not exists pharma_catalog.default.employee_incentive_fact
using delta
location 'abfss://pharmaproject@pgstorageacc.dfs.core.windows.net/gold/'  --and pass here my gold path

### #transfer managed tabl's data to external table

In [0]:


spark.table("pharma_catalog.gold.employee_incentive_fact").write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("pharma_catalog.default.employee_incentive_fact")

In [0]:
%sql
SELECT * FROM pharma_catalog.default.employee_incentive_fact ORDER BY employee_id


employee_id,employee_name,designation,zone,region,city,hq_code,month_key,total_sales,total_quantity,distinct_products_sold,distinct_customers_covered,total_target,achievement_pct,incentive_pct,incentive_amount,incentUpdate_timestamp,curFileSave_timestamp
E100001,Riya Gupta,Product Executive,North,Delhi,Gurugram,HQ-N-127,2025-04,759982.76,135,1,1,0.0,0.0,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100003,Ananya Joshi,Medical Representative,North,Rajasthan,Udaipur,HQ-N-334,2025-08,219940.27,199,1,1,1210215.29,18.17,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100003,Ananya Joshi,Medical Representative,North,Rajasthan,Udaipur,HQ-N-334,2025-08,219940.27,199,1,1,1587587.19,13.85,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100003,Ananya Joshi,Medical Representative,North,Rajasthan,Udaipur,HQ-N-334,2025-08,219940.27,199,1,1,1344230.78,16.36,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100005,Vikram Sharma,Territory Business Executive,East,Jharkhand,Ranchi,HQ-E-897,null,376086.3,77,1,1,292391.75,128.62,15.0,56412.95,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100005,Vikram Sharma,Territory Business Executive,East,Jharkhand,Ranchi,HQ-E-897,2025-04,576224.29,135,1,1,292391.75,197.07,15.0,86433.64,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100006,Naveen Joshi,Area Sales Manager,Central,Chhattisgarh,Raipur,HQ-C-787,2025-09,441112.5,109,1,1,0.0,0.0,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100008,Nisha Nair,Medical Representative,South,Tamil Nadu,Madurai,HQ-S-969,2025-06,257332.05,115,1,1,1329033.07,19.36,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100008,Nisha Nair,Medical Representative,South,Tamil Nadu,Madurai,HQ-S-969,2025-06,257332.05,115,1,1,2160745.38,11.91,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z
E100008,Nisha Nair,Medical Representative,South,Tamil Nadu,Madurai,HQ-S-969,2025-06,257332.05,115,1,1,1889916.22,13.62,0.0,0.0,2026-03-16T03:30:53.985531Z,2026-03-16T03:30:53.985531Z


### #bussiness logics insights

In [0]:

cleaned_employee_incentive_df = spark.sql("""
    with ranked_data as (
        select 
            *,
            date(incentUpdate_timestamp) as incentupdate_date,
            row_number() over (
                partition by employee_id 
                order by date(incentUpdate_timestamp) desc, incentive_amount desc
            ) as rn
        from pharma_catalog.default.employee_incentive_fact
        where employee_id is not null
    )
    select 
        * except(incentUpdate_timestamp, rn),
        incentupdate_date as incentUpdate_timestamp
    from ranked_data
    where rn = 1
""")

# Write the DataFrame to the external table
cleaned_employee_incentive_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pharma_catalog.default.employee_incentive_fact")